# 03 - Feature Analysis

This notebook analyzes feature quality, correlation, cardinality, and feature engineering opportunities.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

data_path = Path(r"C:\Users\User\Desktop\house-price-prediction\data\processed\modeling_dataset.csv")

if not data_path.is_file():
    raise FileNotFoundError(f"File not found: {data_path}")

df = pd.read_csv(data_path)

print("Loaded:", str(data_path))
print("Shape:", df.shape)
display(df.head())

Loaded: C:\Users\User\Desktop\house-price-prediction\data\processed\modeling_dataset.csv
Shape: (167888, 70)


,parcelid,logerror,transactiondate,data_year,airconditioningtypeid,architecturalstyletypeid,basementsqft,bathroomcnt,bedroomcnt,buildingclasstypeid,...,censustractandblock,transaction_year,transaction_month,transaction_quarter,property_age,bath_bed_ratio,avg_room_size,structure_tax_ratio,land_tax_ratio,effective_tax_rate
0,11016594,0.0276,2016-01-01,2016,1.0,NaN,NaN,2.0,3.0,NaN,...,6.037107e+13,2016,1,1,57.0,0.666667,NaN,0.340822,0.659178,0.018702
1,14366692,-0.1684,2016-01-01,2016,NaN,NaN,NaN,3.5,4.0,NaN,...,NaN,2016,1,1,2.0,0.875000,NaN,0.591701,0.408299,0.017340
2,12098116,-0.0040,2016-01-01,2016,1.0,NaN,NaN,3.0,2.0,NaN,...,6.037464e+13,2016,1,1,76.0,1.500000,NaN,0.517022,0.482978,0.095779
3,12643413,0.0218,2016-01-02,2016,1.0,NaN,NaN,2.0,2.0,NaN,...,6.037296e+13,2016,1,1,29.0,1.000000,NaN,0.700417,0.299583,0.012450
4,14432541,-0.0050,2016-01-02,2016,NaN,NaN,NaN,2.5,4.0,NaN,...,6.059042e+13,2016,1,1,35.0,0.625000,285.375,0.390228,0.609772,0.012631


## Missing value ratio by feature

In [2]:
feature_quality = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_ratio": df.isnull().mean().values,
    "n_unique": df.nunique(dropna=True).values
}).sort_values(["missing_ratio", "n_unique"], ascending=[False, False])

feature_quality.head(30)

,column,dtype,missing_ratio,n_unique
9,buildingclasstypeid,float64,0.999815,2
16,finishedsquarefeet13,float64,0.999553,17
6,basementsqft,float64,0.999446,78
44,storytypeid,float64,0.999446,1
49,yardbuildingsqft26,float64,0.999017,112
5,architecturalstyletypeid,float64,0.997212,6
46,typeconstructiontypeid,float64,0.996891,4
19,finishedsquarefeet6,float64,0.995193,649
12,decktypeid,float64,0.992424,1
32,pooltypeid10,float64,0.990315,1


## High-cardinality categorical features

In [3]:
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
cardinality = pd.DataFrame({
    "column": cat_cols,
    "unique_values": [df[c].nunique(dropna=True) for c in cat_cols]
}).sort_values("unique_values", ascending=False)
cardinality.head(20)

,column,unique_values
3,propertyzoningdesc,2346
0,transactiondate,616
2,propertycountylandusecode,90
1,hashottuborspa,2
4,fireplaceflag,2
5,taxdelinquencyflag,1


## Correlation with target

In [4]:
if "logerror" in df.columns:
    corr = df.select_dtypes(include=[np.number]).corr(numeric_only=True)["logerror"].sort_values(key=lambda s: s.abs(), ascending=False)
    corr.head(20)
else:
    print("Target column 'logerror' not found.")

## Highly correlated numerical features

In [5]:
num_df = df.select_dtypes(include=[np.number]).copy()
corr_matrix = num_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0":"feature_1","level_1":"feature_2",0:"correlation"})
    .query("correlation > 0.85")
    .sort_values("correlation", ascending=False)
)
high_corr.head(30)

,feature_1,feature_2,correlation
154,data_year,assessmentyear,1.000000
159,data_year,transaction_year,1.000000
294,bathroomcnt,calculatedbathnbr,1.000000
538,calculatedfinishedsquarefeet,finishedsquarefeet15,1.000000
540,calculatedfinishedsquarefeet,finishedsquarefeet6,1.000000
536,calculatedfinishedsquarefeet,finishedsquarefeet12,1.000000
537,calculatedfinishedsquarefeet,finishedsquarefeet13,1.000000
629,finishedsquarefeet13,regionidneighborhood,1.000000
1504,structure_tax_ratio,land_tax_ratio,1.000000
1420,assessmentyear,transaction_year,1.000000


## Suggested feature engineering ideas

In [6]:
ideas = [
    "Create house_age = transaction_year - yearbuilt",
    "Create room_ratio = calculatedfinishedsquarefeet / roomcnt",
    "Create tax_to_area = taxvaluedollarcnt / calculatedfinishedsquarefeet",
    "Group rare categories in high-cardinality columns",
    "Log-transform heavily skewed tax/value features",
    "Extract year/month from transactiondate"
]
pd.DataFrame({"feature_engineering_ideas": ideas})

,feature_engineering_ideas
0,Create house_age = transaction_year - yearbuilt
1,Create room_ratio = calculatedfinishedsquarefe...
2,Create tax_to_area = taxvaluedollarcnt / calcu...
3,Group rare categories in high-cardinality columns
4,Log-transform heavily skewed tax/value features
5,Extract year/month from transactiondate
